<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 06. word2vec: Skip-gram과 Negative Sampling — Distributed Representations of Words and Phrases and their Compositionality

- **원 논문:** [Distributed Representations of Words and Phrases and their Compositionality](https://arxiv.org/abs/1310.4546)
- **저자 / 발표:** Tomas Mikolov, Ilya Sutskever, Kai Chen, Greg Corrado, Jeffrey Dean · NeurIPS 2013 (2013)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** PyTorch 텐서, 임베딩, 이진 교차 엔트로피, 코사인 유사도

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

작은 문장 모음에서 Skip-gram 중심어-문맥 쌍을 만들고, unigram 분포의 3/4승으로 음성 표본을 뽑아 식 (4)의 SGNS 목적함수를 최적화한다. 손실 감소와 학습된 단어 공간의 코사인 구조를 확인한다.

**원 논문과 다른 것**

논문은 최대 약 10억 단어 규모 말뭉치와 큰 어휘, 다수의 음성 표본으로 단어·구 벡터 및 analogy 정확도를 평가했다. 여기서는 8개 미니 문장과 수십 개 파라미터만 사용하므로 원 논문의 품질 수치는 재현하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, 식 (1): Skip-gram이 주변 단어의 로그확률을 최대화 | `make_skipgram_pairs`가 윈도 안의 (중심어, 문맥어) 양성 쌍을 생성 | 쌍 개수와 자기 자신이 문맥에 들어가지 않는다는 assert |
| §2.2, 식 (4): Negative Sampling 목적함수 | `sgns_loss`가 positive `log σ`와 negative `log σ(-·)`를 계산 | 스칼라·유한 손실 및 모든 임베딩에 대한 gradient assert |
| §2.2: noise distribution에 unigram의 3/4승 사용 | `noise_probs = count**0.75 / Z`의 noise distribution | 확률 합 1과 고빈도 단어 확률의 완화 여부를 검사 |
| §4: 학습된 단어·구 벡터의 의미적 규칙성 평가 | 학습 embedding의 코사인 유사도 행렬과 2차원 SVD 투영 | 손실 감소율, 유사도 범위, 시각화로 미니 재현을 진단 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\mathcal{L}_{\mathrm{SGNS}}=-\log\sigma(v'_w{}^\top v_c)
-\sum_{j=1}^{K}\log\sigma(-v'_{n_j}{}^\top v_c)
$$

- $v_c$는 center, $v'_w$는 positive context, $v'_{n_j}$는 negative word embedding입니다.
- 전체 vocabulary softmax 대신 positive pair와 $K$개 negative pair를 binary 분류합니다.
- window pair 생성, unigram negative sampling, embedding 점수, loss를 따로 구현합니다.
- index $[B]$는 embedding $[B,D]$로 변하고 dot product는 logit $[B]$를 만듭니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, 식 (1): Skip-gram이 주변 단어의 로그확률을 최대화
- **노트북 구현:** `make_skipgram_pairs`가 윈도 안의 (중심어, 문맥어) 양성 쌍을 생성
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 쌍 개수와 자기 자신이 문맥에 들어가지 않는다는 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 최대 약 10억 단어 규모 말뭉치와 큰 어휘, 다수의 음성 표본으로 단어·구 벡터 및 analogy 정확도를 평가했다. 여기서는 8개 미니 문장과 수십 개 파라미터만 사용하므로 원 논문의 품질 수치는 재현하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

Skip-gram은 중심 단어 $w_I$로 주변 단어 $w_O$를 예측한다. 전체 어휘
softmax 대신 Negative Sampling은 실제 쌍에는
$\log\sigma(v'_{w_O}{}^\top v_{w_I})$, 잡음 단어 $w_i$에는
$\log\sigma(-v'_{w_i}{}^\top v_{w_I})$를 준다(논문 식 (4)).

아래 실습에서는 논문이 보고한 `unigram_frequency ** 0.75` 잡음 분포도
그대로 구현한다. 작은 데이터이므로 최종 analogy 점수보다 목적함수와
임베딩 기하가 올바르게 연결되는지가 핵심이다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
from collections import Counter

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 606
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(f"torch={torch.__version__}, seed={SEED}; {ACCELERATOR.summary()}")

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2, 식 (1): Skip-gram이 주변 단어의 로그확률을 최대화 / §2.2: noise distribution에 unigram의 3/4승 사용
- **이 셀의 책임:** `make_skipgram_pairs`가 윈도 안의 (중심어, 문맥어) 양성 쌍을 생성 / `noise_probs = count**0.75 / Z`의 noise distribution
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 쌍 개수와 자기 자신이 문맥에 들어가지 않는다는 assert / 확률 합 1과 고빈도 단어 확률의 완화 여부를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 06-1: window 안의 (중심어, 문맥어) 쌍을 만드세요.
# TODO 06-2: count의 3/4승을 정규화해 noise_probs를 만드세요.
sentences = [
    "king royal palace queen royal palace".split(),
    "queen royal crown king royal crown".split(),
    "man human adult woman human adult".split(),
    "woman human person man human person".split(),
    "paris france capital berlin germany capital".split(),
    "berlin germany capital paris france capital".split(),
    "cat pet animal dog pet animal".split(),
    "dog pet animal cat pet animal".split(),
]
vocab = sorted({token for sentence in sentences for token in sentence})
stoi = {token: index for index, token in enumerate(vocab)}


def make_skipgram_pairs(rows, window=2):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


pairs = make_skipgram_pairs(sentences, window=2)
counts = torch.tensor(
    [Counter((token for row in sentences for token in row))[w] for w in vocab],
    dtype=torch.float32,
)
noise_probs = ...
pairs, counts, noise_probs = ACCELERATOR.move(pairs, counts, noise_probs)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2.2, 식 (4): Negative Sampling 목적함수
- **이 셀의 책임:** `sgns_loss`가 positive `log σ`와 negative `log σ(-·)`를 계산
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 스칼라·유한 손실 및 모든 임베딩에 대한 gradient assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 06-3: 입력/출력 임베딩을 가진 SGNS 모형을 완성하세요.
# TODO 06-4: 논문 식 (4)의 음의 로그우도를 반환하세요.
class SGNS(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, vocab_size, embedding_dim):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, center_ids, positive_ids, negative_ids):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def sgns_loss(positive_scores, negative_scores):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2.2, 식 (4): Negative Sampling 목적함수 / §2.2: noise distribution에 unigram의 3/4승 사용
- **이 셀의 책임:** `sgns_loss`가 positive `log σ`와 negative `log σ(-·)`를 계산 / `noise_probs = count**0.75 / Z`의 noise distribution
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 스칼라·유한 손실 및 모든 임베딩에 대한 gradient assert / 확률 합 1과 고빈도 단어 확률의 완화 여부를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 06-5: 고정된 음성 표본으로 SGNS를 학습하고 loss_history를 채우세요.
torch.manual_seed(SEED)
model = SGNS(len(vocab), embedding_dim=12).to(DEVICE)
generator = torch.Generator().manual_seed(SEED + 1)
negative_ids = ...
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_history = []
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4: 학습된 단어·구 벡터의 의미적 규칙성 평가
- **이 셀의 책임:** 학습 embedding의 코사인 유사도 행렬과 2차원 SVD 투영
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 손실 감소율, 유사도 범위, 시각화로 미니 재현을 진단. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 06-6: 입력/출력 임베딩 평균으로 코사인 행렬과 2D 투영을 그리세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 해석 체크

- 왜 출력 임베딩이 입력 임베딩과 별도로 필요한가?
- `0.75`를 `1.0`으로 바꾸면 자주 등장하는 단어가 음성 표본에서 어떻게 달라지는가?
- 이 작은 투영이 원 논문의 analogy 정확도를 재현했다고 말할 수 없는 이유를 적어 보세요.

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.